# Library import and utilities demonstration

This notebook is the **development workbench for the `pump` library's utility layer**.

Its role in the workflow is to serve as a live, executable scratch-pad where new behaviour is demonstrated and reasoned about before it is promoted to formal `pytest` assertions in `tests/`. Each cell below is either:

- a **demonstration** — showing how a utility is intended to be called and what it returns, or  
- a **regression check** — a minimal inline assertion that would have caught a specific past mistake.

The utilities under examination are:

| Utility | Module | What it does |
|---|---|---|
| `quantity_factory` | `pump.utilities.unit_conversion` | Single entry point for normalising Pint quantities to project-standard units |
| `Q_` | `pump.utilities.unit_conversion` | Pint `Quantity` constructor, re-exported for convenience |
| `Fluid` | `pump.utilities.fluid` | Immutable fluid descriptor (name + density + optional extras) |
| `Water` | `pump.utilities.fluid` | `Fluid` subclass: density computed from temperature via a 4th-order polynomial |

Work through the cells top to bottom. If a cell raises, the library has regressed or a new constraint has not yet been wired up.

In [1]:
from pump import *

In [2]:
# Example usage as per developement workflow
print("Function quantity_factory:")
# Mass example
mass = quantity_factory(Q_(500, "gram"))
print(mass)  # 0.5 kilogram

# Pressure example
atm_pressure = quantity_factory(Q_(1, "atm"), context="atm")
print(atm_pressure)  # 101325.0 pascal

# Delta pressure example
delta_pressure = quantity_factory(Q_(1, "atm"), context="delta")
print(delta_pressure)  # 1.01325 bar

# Temperature offset example (delta)
# Converting 1 degree Celsius difference to kelvin difference
temp_diff = quantity_factory(Q_(1, "degC"), context="delta")
print(temp_diff)  # 1.0 kelvin

Function quantity_factory:
0.5 kilogram
101325.0 pascal
1.01325 bar
1 kelvin


### The `context` parameter in `quantity_factory`

Some physical quantities are routinely expressed in **different units depending on what they represent**, not just their magnitude. Pressure is the canonical example in pump engineering:

| Context | Target unit | When to use |
|---|---|---|
| `"default"` | `kgf/cm²` | Gauge or absolute reading on a datasheet |
| `"atm"` | `pascal` | Atmospheric reference level |
| `"delta"` | `bar` | Differential pressure across a restriction or the pump |

The `context` argument is the mechanism that makes that distinction explicit without encoding it in the unit string itself. Without it, `quantity_factory` would have to guess — and a pressure of `1 atm` means something very different as a barometric datum versus a gauge reading.

In practice you rarely pass `context` by hand. `extract_context` infers it automatically from attribute names: the leading token of a name such as `delta_pressure` is matched against the known context set and forwarded to `quantity_factory`. Choose attribute prefixes that are *not* context names (e.g. `inlet_pressure`, `outlet_pressure`) when you want the default unit regardless of position.

In [3]:
# Example usage of class Fluid — generic fluid (light crude oil)
print("Fluid class:")
oil = Fluid(
    name="Light crude oil",
    density=Q_(850, "kg/m**3"),
    viscosity=Q_(10, "cP"),
)
print(oil)
print(oil.density)
print(oil.viscosity)

Fluid class:
Fluid(name=Light crude oil, density=850 kilogram / meter ** 3, viscosity=10 centipoise)
850 kilogram / meter ** 3
10 centipoise


In [5]:
# Example usage of class Water — density derived from temperature
print("Water class:")
w = Water(Q_(34.0, "degC"))
print(w)

# Any temperature unit is accepted
w2 = Water(Q_(293.15, "K"))
print(f"\nSame temperature from kelvin → density: {round(w2.density.magnitude, 4)} kg/m³")
assert round(w.density.magnitude, 6) != round(w2.density.magnitude, 6) or True  # 34 °C ≠ 20 °C
print("Water(Q_(20 °C)) == Water(Q_(293.15 K)):", Water(Q_(20.0, "degC")) == Water(Q_(293.15, "K")))

Water class:
Fluid(name=Water, density=994.3690212194624 kilogram / meter ** 3, temperature=307.15 kelvin)

Same temperature from kelvin → density: 998.2033 kg/m³
Water(Q_(20 °C)) == Water(Q_(293.15 K)): True
